In [41]:
import pandas as pd
import numpy as np
import re
import os
import torch
from datasets import Dataset
from transformers import AutoTokenizer

In [42]:
df = pd.read_csv('../data/cleaned/dataset_cleaned.csv')
print('Data file loaded')

Data file loaded


In [43]:
try:
    from Siamese import BurmeseConverter
except ImportError:
    from Siamese import BurmeseConverter

print("Loading configurations and downloading tokenizer...")
CSV_FILE_PATH = "../data/cleaned/dataset_cleaned.csv"
MODEL_NAME = "xlm-roberta-base"

converter = BurmeseConverter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def clean_and_syllable_map(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""
    
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    
    def tokenize_myanmar_chunk(match):
        myanmar_text = match.group(0)
        try:
            result = converter.syllable_tokenization(1, myanmar_text)
            if result.startswith("With the virama mark:"):
                result = result.replace("With the virama mark:", "").strip()
            return " " + result + " "
        except Exception:
            return myanmar_text

    processed_text = re.sub(r'[\u1000-\u109F\uAA60-\uAA7F]+', tokenize_myanmar_chunk, text)
    processed_text = re.sub(r'\s+', ' ', processed_text).strip()
    
    return processed_text

if not os.path.exists(CSV_FILE_PATH):
    raise FileNotFoundError(f"Could not locate '{CSV_FILE_PATH}' in your current directory.")

print(f"Reading dataset from {CSV_FILE_PATH}...")
df = pd.read_csv(CSV_FILE_PATH)

print(f"Dataset loaded. Total rows: {len(df)}")
print("Running text syllable cleaning engine...")
df['cleaned_text'] = df['text'].astype(str).apply(clean_and_syllable_map)

print("Converting table matrix to Hugging Face mapping dataset...")
hf_dataset = Dataset.from_pandas(df[['cleaned_text', 'emotion_class']])

def batch_tokenize_step(examples):
    return tokenizer(
        examples["cleaned_text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

print("Executing SentencePiece subword tokenization across data batches...")
tokenized_dataset = hf_dataset.map(batch_tokenize_step, batched=True)

tokenized_dataset = tokenized_dataset.rename_column("emotion_class", "labels")
tokenized_dataset = tokenized_dataset.remove_columns(["cleaned_text"])
tokenized_dataset.set_format("torch")


Loading configurations and downloading tokenizer...
Reading dataset from ../data/cleaned/dataset_cleaned.csv...
Dataset loaded. Total rows: 1890
Running text syllable cleaning engine...
Converting table matrix to Hugging Face mapping dataset...
Executing SentencePiece subword tokenization across data batches...


Map:   0%|          | 0/1890 [00:00<?, ? examples/s]

In [44]:
df.columns

Index(['text', 'emotion_class', 'cleaned_text'], dtype='str')

In [45]:
df = df.drop(columns=['text'])

In [46]:
df.head(5)

,emotion_class,cleaned_text
0,joy,ငြိမ်း ချမ်း တဲ့ စန် ဒ ယား သံ စဉ် ထဲ မှာ တွေ့ ...
1,joy,ဒီ စ နေ ၊ တ နင်္ ဂ နွေ မိ သား စု နဲ့ အ တူ အ ရည...
2,suprise,အ ခက် အ ခဲ တွေ ကြား က နေ မ ထင် မှတ် ဘဲ အ ဖြေ ရ...
3,joy,သက် ကြီး ရွယ် အို များ အ တွက် စာ အုပ် က လပ် တစ...
4,love,မင်း ကို စောင့် ရှောက် ခွင့် ရ တာ ငါ့ ဘ ဝ ရဲ့ ...


In [47]:
df.columns

Index(['emotion_class', 'cleaned_text'], dtype='str')

In [48]:
df['emotion_class'].value_counts()

emotion_class
neutral    329
joy        303
sadness    264
love       256
suprise    251
anger      250
fear       234
Name: count, dtype: int64

In [49]:
df.to_csv('../data/processed/dataset_processed.csv', index=False)
print("Saved")

Saved
